### Setup

In [1]:
import numpy as np
import choix
from scipy.optimize import minimize
import scipy.stats as stats
import matplotlib.pyplot as plt
import random
from matplotlib import colors
import pandas as pd
import seaborn as sns
import pickle
import os
import sys

sys.path.append(os.path.abspath('../../'))
from metrics import compute_acc, compute_weighted_acc
from opt_fair import *
from distribution_utils import safe_kendalltau, to_numpy

In [2]:
!nvidia-smi

Sat Jun 27 05:49:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:17:00.0 Off |                    0 |
| N/A   74C    P0            271W /  300W |   34971MiB /  81920MiB |    100%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')
print(device)

cuda


In [4]:
print(f"Current PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")

Current PyTorch version: 2.4.0a0+f70bd71a48.nv24.06
CUDA available: True
CUDA version: 12.5


In [5]:
with open("data/PassagePC.pickle", 'rb') as handle:
    PC_faceage = pickle.load(handle)    
with open("data/PassageDF.pickle", 'rb') as handle:
    df_faceage = pickle.load(handle)

In [6]:
df_faceage

,label,score
0,"a star. Our planet, Earth, orbits, or circles,...",1
1,"Adam, We did not have plastic toys. I played w...",1
2,Who said the little owl. Who wants to hunt wit...,1
3,dead leaf. This is a mole. Moles burrow underg...,1
4,ereaddatagradepsenvironcomp.html Environment r...,1
...,...,...
467,work over the summer on any changes they wish ...,12
468,between January and December plunged the Unite...,12
469,into a newly opened bank account. I was amazed...,12
470,"occurring phenomenon, manmade by products are ...",12


In [7]:
import opt_fair
all_pc_faceage  = opt_fair._pc_without_reviewers(PC_faceage)

size = len(df_faceage)
print(size)
classes = [0]*size

472


In [8]:
print(len(all_pc_faceage))

11763


In [9]:
gt_scores = to_numpy(df_faceage['score'].tolist())

In [10]:
max_iter=30000

## Extra Ablation

### PG EM - flipped orders

In [11]:
import random
from pgem_initialize_beta_1 import EMWrapper_random_r

In [12]:
import time

PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []
pgem_time = []
for sd in range(10):
    start = time.time()
    pg = EMWrapper_random_r(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    pgem_time.append(end-start)

    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda


  0%|                                                                               | 2/30000 [00:00<2:12:14,  3.78it/s]

Iter 000: Log-likelihood = -0.451188


  0%|▏                                                                             | 58/30000 [00:09<1:23:00,  6.01it/s]


Converged at iter 58, ΔLL = 9.834766e-07
cuda


...

Converged at iter 58, ΔLL = 8.940697e-07
cuda


  0%|                                                                               | 1/30000 [00:00<1:00:59,  8.20it/s]

Iter 000: Log-likelihood = -0.451880


  0%|▏                                                                             | 58/30000 [00:07<1:05:12,  7.65it/s]

Converged at iter 58, ΔLL = 9.536743e-07


In [13]:
print(f"{np.mean(pgem_time)} +- {np.std(pgem_time)}")

8.402139019966125 +- 0.894896057942999


In [14]:
PGEM_accs

[0.6958720684051514,
 0.6958720684051514,
 0.6958720684051514,
 0.6958720684051514,
 0.6958720684051514,
 0.6958720684051514,
 0.6958720684051514,
 0.6958821415901184,
 0.6958821415901184,
 0.6958720684051514]

In [15]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.6958740830421448 ± 4.029273986816407e-06, Weighted Accuracy: 0.7555751919746398 ± 6.103515625e-06, Kendall's Tau: 0.36981380902965216 ± 7.623965922887165e-06


### PG EM - flipped orders (r then beta)

In [16]:
import random
from pgem_initialize_beta_1 import EMWrapper_random_r

In [17]:
import time

PGEM_accs, PGEM_waccs, PGEM_taus = [], [], []
pgem_time = []
for sd in range(10):
    start = time.time()
    pg = EMWrapper_random_r(PC_faceage, max_iter, device, sd)
    r_est, beta_est, ll = pg.run_algorithm()
    end = time.time()
    pgem_time.append(end-start)

    if np.isnan(r_est).any() or np.isnan(beta_est).any() or np.isnan(ll):
        print("Skipping nan")
        continue
    
    r_est_np = to_numpy(r_est)
    
    gt_scores = to_numpy(df_faceage['score'].tolist())
    current_tau = safe_kendalltau(r_est_np, gt_scores)
    if current_tau < 0:
        r_est_np = -r_est_np
    pgem_acc = compute_acc(df_faceage, 1*r_est_np, device)
    pgem_wacc = compute_weighted_acc(df_faceage, 1*r_est_np, device)
    pgem_tau = safe_kendalltau(r_est_np, gt_scores)
    
    PGEM_accs.append(pgem_acc)    
    PGEM_waccs.append(pgem_wacc)    
    PGEM_taus.append(pgem_tau)

cuda


  0%|                                                                               | 1/30000 [00:00<1:27:14,  5.73it/s]

Iter 000: Log-likelihood = -0.451188


  0%|▏                                                                             | 58/30000 [00:08<1:12:56,  6.84it/s]


Converged at iter 58, ΔLL = 9.834766e-07
cuda


...

Converged at iter 58, ΔLL = 8.940697e-07
cuda


  0%|                                                                               | 1/30000 [00:00<1:00:29,  8.27it/s]

Iter 000: Log-likelihood = -0.451880


  0%|▏                                                                             | 58/30000 [00:08<1:14:37,  6.69it/s]

Converged at iter 58, ΔLL = 9.536743e-07


In [18]:
print(f"{np.mean(pgem_time)} +- {np.std(pgem_time)}")

8.515735316276551 +- 0.4864554070605709


In [19]:
PGEM_accs

[0.6958720684051514,
 0.6958720684051514,
 0.6958720684051514,
 0.6958720684051514,
 0.6958821415901184,
 0.6958720684051514,
 0.6958720684051514,
 0.6958821415901184,
 0.6958821415901184,
 0.6958720684051514]

In [20]:
print(f"PGEM -- Accuracy: {np.mean(PGEM_accs)} ± {np.std(PGEM_accs)}, Weighted Accuracy: {np.mean(PGEM_waccs)} ± {np.std(PGEM_waccs)}, Kendall's Tau: {np.mean(PGEM_taus)} ± {np.std(PGEM_taus)}")

PGEM -- Accuracy: 0.6958750903606414 ± 4.6161132600756705e-06, Weighted Accuracy: 0.7555767178535462 ± 6.99245558922705e-06, Kendall's Tau: 0.36981571502113286 ± 8.734350234348574e-06
